# NBA Shot Logs 2014–15 — Starter Notebook

Every tracked shot from the first part of the 2014–15 NBA season (about 128,000 shots by 281 players). Each row is **one shot**: who took it, how far from the basket, how close the nearest defender was, how much time was on the shot clock, and whether it went in.

**How this notebook is organized**

| Section | What it does |
|---|---|
| Settings | Where the data lives (the only cell you might edit) |
| Load Data | Imports libraries, loads and prepares the data |
| Inspect the Dataset | First rows, size, column types |
| Describe Variables | Summary statistics |
| Check Missing Data | What's missing and why it matters |
| Basic Visualizations | A few plots to get oriented |
| Variables You May Want to Explore | Outcomes, predictors, grouping and time variables |
| Your Analysis Starts Here | Your space for HW 1, HW 2, and HW 3 |

Cells marked 🧰 are **starter code** written by the instructor. Cells marked ✏️ are **yours**.

Open the table of contents (the ☰ icon on the left in Colab) to jump between sections.

## Settings

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

If the default link stops working, or you downloaded the file yourself, change the path below. See the dataset guide for where to get the data.

In [ ]:
# Settings: where the data lives.
# By default the data load from this course's GitHub repository, so there's nothing to download.
REPO = "cogs118d/datasets"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/data/03_nba_shots/"

# Cloned the repo and working on your own computer? Use the local folder instead:
# BASE_URL = "../data/03_nba_shots/"
# Saved the files to Google Drive? Point BASE_URL at that folder, e.g.:
# BASE_URL = "/content/drive/MyDrive/03_nba_shots/"
DATA_URL = BASE_URL + "shot_logs.csv"

# Load Data

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# Libraries used in every COGS 118D starter notebook
import numpy as np                        # numbers and arrays
import pandas as pd                       # data tables ("DataFrames")
import matplotlib.pyplot as plt           # plots
import statsmodels.formula.api as smf     # GLMs and mixed models (you'll use this later)

pd.set_option("display.max_columns", 60)  # show more columns when printing tables
plt.rcParams["figure.figsize"] = (7, 4)   # default plot size

In [ ]:
df = pd.read_csv(DATA_URL)
print("Loaded", df.shape[0], "shots")

## Preprocessing

A few values are impossible or unusual. We flag them rather than deleting rows, so you can decide what to do.

In [ ]:
# Touch time (seconds the shooter held the ball) can't be negative; set negative values to missing
print("Negative touch times:", (df["TOUCH_TIME"] < 0).sum())
df.loc[df["TOUCH_TIME"] < 0, "TOUCH_TIME"] = np.nan

# The shot clock is off when less than 24 seconds remain in a period, so SHOT_CLOCK is missing then
print("Missing shot clock:", df["SHOT_CLOCK"].isna().sum())

# Readable versions of a few columns
df["home"] = (df["LOCATION"] == "H").astype(int)                 # 1 = shooter's team at home
df["three_pointer"] = (df["PTS_TYPE"] == 3).astype(int)
df["game_date"] = pd.to_datetime(df["MATCHUP"].str.split(" - ").str[0], format="%b %d, %Y")
df[["player_name", "SHOT_DIST", "CLOSE_DEF_DIST", "SHOT_CLOCK", "FGM"]].head()

# Inspect the Dataset

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# The first 5 rows: each row is one observation
df.head()

In [ ]:
# Dimensions: (number of rows, number of columns)
print("Rows, columns:", df.shape)

In [ ]:
# Column names and data types (numbers, text, dates, True/False)
df.info()

# Describe Variables

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
df[["SHOT_DIST", "CLOSE_DEF_DIST", "SHOT_CLOCK", "DRIBBLES", "TOUCH_TIME", "FGM", "PTS"]].describe().round(2)

In [ ]:
print("Overall make rate:", round(df["FGM"].mean(), 3))
print(df["PTS_TYPE"].value_counts())

# Check Missing Data

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# Count and percent of missing values in each column (only columns with any missing)
missing = df.isna().sum()
missing_table = pd.DataFrame({"n_missing": missing, "percent": (100 * missing / len(df)).round(1)})
missing_table[missing_table["n_missing"] > 0].sort_values("n_missing", ascending=False)

**What to notice.** `SHOT_CLOCK` is missing for about 4% of shots (end-of-period possessions). A few hundred touch times were negative and are now missing. Some shots labeled 3-pointers have short recorded distances, a sign of tracking error.

# Basic Visualizations

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

These plots are for getting oriented. They are not answers to any homework question.

In [ ]:
# Make rate by shot distance (1-foot bins)
df["dist_bin"] = df["SHOT_DIST"].round()
by_dist = df[df["SHOT_DIST"] <= 35].groupby("dist_bin")["FGM"].mean()
plt.plot(by_dist.index, by_dist.values, marker=".")
plt.xlabel("Shot distance (feet)")
plt.ylabel("Proportion made")
plt.title("Make rate by distance")
plt.show()

In [ ]:
# Make rate by distance to the closest defender (1-foot bins, up to 15 ft)
df["def_bin"] = df["CLOSE_DEF_DIST"].round()
by_def = df[df["CLOSE_DEF_DIST"] <= 15].groupby("def_bin")["FGM"].mean()
plt.plot(by_def.index, by_def.values, marker=".")
plt.xlabel("Distance to closest defender (feet)")
plt.ylabel("Proportion made")
plt.title("Make rate by defender distance")
plt.show()

In [ ]:
# Where shots are taken from
plt.hist(df["SHOT_DIST"], bins=48, edgecolor="white")
plt.xlabel("Shot distance (feet)")
plt.ylabel("Number of shots")
plt.title("Shot distances")
plt.show()

In [ ]:
# How many shots each player took
plt.hist(df.groupby("player_id").size(), bins=30, edgecolor="white")
plt.xlabel("Shots per player")
plt.ylabel("Number of players")
plt.title("Shots per player")
plt.show()

# Variables You May Want to Explore

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

**Outcome variables:** `FGM` (made = 1, missed = 0), `PTS` (0, 2, or 3).

**Predictors:** `SHOT_DIST`, `CLOSE_DEF_DIST`, `SHOT_CLOCK`, `DRIBBLES`, `TOUCH_TIME`, `home`, `PERIOD`, `three_pointer`, and game context (`W`, `FINAL_MARGIN`; these are known only after the game, so be careful).

In [ ]:
outcome_vars = ["FGM", "PTS"]
predictor_vars = ["SHOT_DIST", "CLOSE_DEF_DIST", "SHOT_CLOCK", "DRIBBLES", "TOUCH_TIME", "home", "PERIOD"]
df[outcome_vars + predictor_vars].describe().round(2)

## Grouping structure (for mixed-effects models in HW 3)

Shots are grouped within **shooters** (`player_id`), within **games** (`GAME_ID`), and within the **closest defender** (`CLOSEST_DEFENDER_PLAYER_ID`). Shooters and defenders are *crossed* grouping factors.

In [ ]:
print("Shooters:", df["player_id"].nunique(), "| median shots per shooter:", df.groupby("player_id").size().median())
print("Games:", df["GAME_ID"].nunique(), "| median shots per game:", df.groupby("GAME_ID").size().median())
print("Defenders:", df["CLOSEST_DEFENDER_PLAYER_ID"].nunique())

## Time-to-event variables (for survival analysis in HW 3)

**Not appropriate for this dataset.** Nothing here measures the time until an event. (The shot clock is a time, but every shot happens by definition, so there is no censoring or waiting-time question.) For HW 3, use a **mixed-effects model** with shots nested in players and games.

---
# Your Analysis Starts Here

> ✏️ **Your analysis.** Everything below this line is yours. Add as many cells as you need.

The dataset guide lists starter questions for each homework. 

## HW 1 — Describe

In [ ]:
# ✏️ Your HW 1 code here


## HW 2 — Predict / Moderate / Mediate

In [ ]:
# ✏️ Your HW 2 code here


## HW 3 — Mixed Effects / Survival

In [ ]:
# ✏️ Your HW 3 code here
